# Pneumonia Detection using Chest X-Ray Images

Trains a CNN (transfer learning with VGG16) to classify chest X-rays as **NORMAL** or **PNEUMONIA**.

Dataset: [chest-xray-pneumonia on Kaggle](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

**Before running:** In Colab, go to `Runtime > Change runtime type` and select **GPU** for much faster training.

## 1. Install / import libraries

In [ ]:
!pip install -q opencv-python-headless kaggle scikit-learn seaborn

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Download the dataset from Kaggle

1. Go to https://www.kaggle.com/settings -> "Create New Token" to download `kaggle.json`.
2. Run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content
!unzip -q /content/chest-xray-pneumonia.zip -d /content

DATASET_DIR = '/content/chest_xray'
print(os.listdir(DATASET_DIR))

## 3. (Optional) Mount Google Drive to save the trained model persistently

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_DIR = '/content/drive/MyDrive/pneumonia_detection_models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 4. Configuration

In [ ]:
IMG_SIZE = 224
CHANNELS = 3
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 1e-4
SEED = 42
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'val')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

## 5. Look at a few sample images with OpenCV

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(CLASS_NAMES):
    folder = os.path.join(TRAIN_DIR, cls)
    sample_file = os.listdir(folder)[0]
    img = cv2.imread(os.path.join(folder, sample_file))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i * 2].imshow(img_rgb)
    axes[i * 2].set_title(f'{cls} (original)')
    axes[i * 2].axis('off')

    resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    axes[i * 2 + 1].imshow(resized)
    axes[i * 2 + 1].set_title(f'{cls} (resized {IMG_SIZE}x{IMG_SIZE})')
    axes[i * 2 + 1].axis('off')
plt.tight_layout()
plt.show()

## 6. Data generators (augmentation on train only)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=CLASS_NAMES, seed=SEED, shuffle=True)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=CLASS_NAMES, seed=SEED, shuffle=False)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=CLASS_NAMES, seed=SEED, shuffle=False)

labels = train_generator.classes
class_weights = dict(zip(np.unique(labels), compute_class_weight('balanced', classes=np.unique(labels), y=labels)))
print('Class weights:', class_weights)

## 7. Build the model (VGG16 transfer learning)

In [ ]:
def build_transfer_model(input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs, name='transfer_vgg16')
    model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='binary_crossentropy',
                  metrics=['accuracy', 'AUC', 'Precision', 'Recall'])
    return model

model = build_transfer_model()
model.summary()

## 8. Train

In [ ]:
BEST_MODEL_PATH = '/content/best_model.h5'

callbacks = [
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

## 9. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(history.history['accuracy'], label='train_acc')
axes[0].plot(history.history['val_accuracy'], label='val_acc')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train_loss')
axes[1].plot(history.history['val_loss'], label='val_loss')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Evaluate on the test set

In [ ]:
test_generator.reset()
y_prob = model.predict(test_generator, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_generator.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend()
plt.show()

## 11. Grad-CAM on a single test image (using OpenCV)

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=0):
    grad_model = tf.keras.models.Model(
        [model.get_layer('vgg16').input],
        [model.get_layer('vgg16').get_layer(last_conv_layer_name).output]
    )
    with tf.GradientTape() as tape:
        conv_outputs = grad_model(img_array)
        tape.watch(conv_outputs)
        preds = model(img_array)
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Grab one test image
sample_class = 'PNEUMONIA'
sample_folder = os.path.join(TEST_DIR, sample_class)
sample_file = os.listdir(sample_folder)[0]
sample_path = os.path.join(sample_folder, sample_file)

img_bgr = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)).astype('float32') / 255.0
img_batch = np.expand_dims(img_resized, axis=0)

prob = float(model.predict(img_batch, verbose=0)[0][0])
print(f'Prediction: {"PNEUMONIA" if prob >= 0.5 else "NORMAL"} (score={prob:.4f})')

heatmap = make_gradcam_heatmap(img_batch, model, 'block5_conv3')
heatmap_resized = cv2.resize(heatmap, (img_bgr.shape[1], img_bgr.shape[0]))
heatmap_uint8 = np.uint8(255 * heatmap_resized)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
overlay = cv2.addWeighted(heatmap_color, 0.4, img_bgr, 0.6, 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
plt.show()

## 12. Save the final model to Google Drive

In [ ]:
final_path = os.path.join(MODEL_SAVE_DIR, 'best_model.h5')
model.save(final_path)
print(f'Model saved to: {final_path}')
print('Download this file and place it in your local project as models/best_model.h5')
print('to use it with src/predict.py, src/evaluate.py, or the Flask app.')